# IC50 Threshold-Cutoff Analysis

Reproduces the threshold-sensitivity and selectivity-ratio analysis built to
address the reviewer comment on the 1000 nM activity cutoff. Covers:

1. Baseline AA/AI/IA/II class distribution at the 1000 nM threshold.
2. The literal reviewer example -- how many molecules sit within a narrow band
   (±1%/5%/10%) of the cutoff on either axis, and would flip class under a tiny
   shift (e.g. 990 vs. 1010 nM).
3. Broader threshold-sensitivity -- how many molecules get relabeled at several
   alternative cutoffs (100 nM to 10,000 nM).
4. Selectivity-ratio validation -- does log10(TDO IC50 / IDO1 IC50) actually
   separate by class in the expected direction, with Kruskal-Wallis and
   Mann-Whitney tests.

**Input required:** `ido1_tdo_common_1126.csv` in the same folder as this notebook.

**Outputs produced:**
- `threshold_sensitivity.csv`
- `common_1126_with_labels_and_ratio.csv`
- `Figure_threshold_scatter.png`
- `Figure_selectivity_ratio_boxplot.png`
- `Figure_threshold_sensitivity_bar.png`

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

%matplotlib inline

df = pd.read_csv('ido1_tdo_common_1126.csv', na_values=['na'])
THRESHOLD = 1000.0  # nM, the manuscript's cutoff
print(f"Loaded {len(df)} molecules")

## 2. Label construction and baseline class distribution

In [ ]:
def make_labels(df, threshold):
    ido_active = df["ido1_ic50"] < threshold
    tdo_active = df["tdo_ic50"] < threshold
    labels = np.select(
        [ido_active & tdo_active, ido_active & ~tdo_active, ~ido_active & tdo_active, ~ido_active & ~tdo_active],
        ["AA", "AI", "IA", "II"],
        default="NA"
    )
    return labels

df["label"] = make_labels(df, THRESHOLD)

print("Baseline class distribution (threshold = 1000 nM):")
print(df["label"].value_counts().reindex(["AA","AI","IA","II"]))
print(f"Total: {len(df)}")

## 3. The literal reviewer example: borderline molecules near the cutoff

How many molecules sit within a narrow band (±1%, ±5%, ±10%) of 1000 nM on
*either* axis -- i.e., molecules whose class label is directly sensitive to a
small shift in the threshold, such as the reviewer's 990 vs. 1010 nM example.

In [ ]:
for band_pct, label in [(1, "+/-1% (990-1010 nM style)"), (5, "+/-5%"), (10, "+/-10%")]:
    lo, hi = THRESHOLD * (1 - band_pct/100), THRESHOLD * (1 + band_pct/100)
    near_ido = df["ido1_ic50"].between(lo, hi)
    near_tdo = df["tdo_ic50"].between(lo, hi)
    near_either = near_ido | near_tdo
    print(f"{label}: band=[{lo:.0f},{hi:.0f}] nM -> "
          f"{near_ido.sum()} near IDO1 cutoff, {near_tdo.sum()} near TDO cutoff, "
          f"{near_either.sum()} molecules ({near_either.sum()/len(df)*100:.1f}%) with EITHER axis borderline")

## 4. Threshold sensitivity: relabeling at alternative cutoffs

Separate from the narrow-band question above: how much would the *overall*
class distribution change if a substantially different threshold (10x smaller
or larger) had been chosen instead?

In [ ]:
alt_thresholds = [100, 300, 500, 1000, 2000, 5000, 10000]
baseline_labels = df["label"].values
sensitivity_rows = []
for t in alt_thresholds:
    alt_labels = make_labels(df, t)
    n_changed = (alt_labels != baseline_labels).sum()
    counts = pd.Series(alt_labels).value_counts().reindex(["AA","AI","IA","II"]).fillna(0).astype(int)
    sensitivity_rows.append({
        "threshold_nM": t, "n_changed_vs_1000nM": n_changed,
        "pct_changed": n_changed/len(df)*100,
        "AA": counts["AA"], "AI": counts["AI"], "IA": counts["IA"], "II": counts["II"]
    })
sens_df = pd.DataFrame(sensitivity_rows)
print(sens_df.to_string(index=False))
sens_df.to_csv("threshold_sensitivity.csv", index=False)
print("\nSaved: threshold_sensitivity.csv")

## 5. Selectivity-ratio validation

Does the discrete AA/AI/IA/II scheme actually track genuine selectivity, or
could borderline double-threshold cases dominate it? Computes the continuous
selectivity ratio log10(TDO IC50 / IDO1 IC50) and checks it against the labels.

In [ ]:
df["log_selectivity_ratio"] = np.log10(df["tdo_ic50"] / df["ido1_ic50"])
# positive = more potent (lower IC50) against IDO1 = IDO-selective direction
# negative = more potent against TDO = TDO-selective direction

print("Selectivity ratio [log10(TDO_IC50/IDO1_IC50)] by class:")
print(df.groupby("label")["log_selectivity_ratio"].describe()[["count","mean","std","50%","min","max"]])

groups = [df.loc[df["label"]==c, "log_selectivity_ratio"].values for c in ["AA","AI","IA","II"]]
h_stat, p_val = stats.kruskal(*groups)
print(f"\nKruskal-Wallis test across AA/AI/IA/II selectivity ratios: H={h_stat:.2f}, p={p_val:.2e}")

ai_ratio = df.loc[df["label"]=="AI", "log_selectivity_ratio"]
ia_ratio = df.loc[df["label"]=="IA", "log_selectivity_ratio"]
u_stat, p_mw = stats.mannwhitneyu(ai_ratio, ia_ratio, alternative="two-sided")
print(f"AI vs IA Mann-Whitney U test: U={u_stat:.1f}, p={p_mw:.2e}")
print(f"AI mean ratio: {ai_ratio.mean():.2f} (positive = IDO-potency bias, as expected for \'IDO-selective\')")
print(f"IA mean ratio: {ia_ratio.mean():.2f} (negative = TDO-potency bias, as expected for \'TDO-selective\')")

weak_ai = (ai_ratio.abs() < np.log10(2)).sum()
weak_ia = (ia_ratio.abs() < np.log10(2)).sum()
print(f"\nAI compounds with <2-fold IDO/TDO potency difference despite \'selective\' label: "
      f"{weak_ai}/{len(ai_ratio)} ({weak_ai/len(ai_ratio)*100:.1f}%)")
print(f"IA compounds with <2-fold IDO/TDO potency difference despite \'selective\' label: "
      f"{weak_ia}/{len(ia_ratio)} ({weak_ia/len(ia_ratio)*100:.1f}%)")

df.to_csv("common_1126_with_labels_and_ratio.csv", index=False)
print("\nSaved: common_1126_with_labels_and_ratio.csv")

## 6. Figures

In [ ]:
COLORS = {"AA": "#8b5cf6", "AI": "#2563eb", "IA": "#0ea5e9", "II": "#94a3b8"}
plt.rcParams.update({"font.size": 11, "figure.facecolor": "white"})

### 6a. IDO1 vs. TDO IC50 scatter, with threshold lines

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5))
for cls, color in COLORS.items():
    sub = df[df["label"] == cls]
    ax.scatter(sub["ido1_ic50"], sub["tdo_ic50"], s=14, alpha=0.6, color=color, label=cls, edgecolors="none")

ax.axvline(THRESHOLD, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(THRESHOLD, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.axvspan(THRESHOLD*0.9, THRESHOLD*1.1, color="red", alpha=0.08)
ax.axhspan(THRESHOLD*0.9, THRESHOLD*1.1, color="red", alpha=0.08)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("IDO1 IC$_{50}$ (nM)")
ax.set_ylabel("TDO IC$_{50}$ (nM)")
ax.set_title("Class assignment vs. the 1000 nM threshold\n(red band = \u00b110% of cutoff on either axis)")
ax.legend(loc="upper left", frameon=True, fontsize=10)
ax.grid(alpha=0.2, which="both")
plt.tight_layout()
plt.savefig("Figure_threshold_scatter.png", dpi=220, bbox_inches="tight")
plt.show()

### 6b. Selectivity ratio by class

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
order = ["IA", "II", "AA", "AI"]
data_by_class = [df.loc[df["label"] == c, "log_selectivity_ratio"].values for c in order]
bp = ax.boxplot(data_by_class, tick_labels=order, patch_artist=True, showfliers=True,
                 medianprops=dict(color="black"))
for patch, cls in zip(bp["boxes"], order):
    patch.set_facecolor(COLORS[cls])
    patch.set_alpha(0.7)

ax.axhline(0, color="black", linestyle="-", linewidth=0.8)
ax.axhline(np.log10(2), color="gray", linestyle=":", linewidth=1)
ax.axhline(-np.log10(2), color="gray", linestyle=":", linewidth=1)
ax.text(4.55, np.log10(2), " 2-fold", va="bottom", fontsize=8.5, color="gray")
ax.text(4.55, -np.log10(2), " 2-fold", va="top", fontsize=8.5, color="gray")

ax.set_ylabel(r"Selectivity ratio: $\log_{10}$(TDO IC$_{50}$ / IDO1 IC$_{50}$)")
ax.set_title("Selectivity ratio by assigned class\n(positive = IDO1-potency-biased, negative = TDO-potency-biased)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figure_selectivity_ratio_boxplot.png", dpi=220, bbox_inches="tight")
plt.show()

### 6c. Threshold-sensitivity bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
colors_bar = ["#ef4444" if t != 1000 else "#16a34a" for t in sens_df["threshold_nM"]]
ax.bar(range(len(sens_df)), sens_df["pct_changed"], color=colors_bar)
ax.set_xticks(range(len(sens_df)))
ax.set_xticklabels([f"{t:,.0f}" for t in sens_df["threshold_nM"]])
ax.set_xlabel("Alternative threshold (nM)")
ax.set_ylabel("% of molecules relabeled\nvs. 1000 nM baseline")
ax.set_title("Class-label sensitivity to threshold choice")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figure_threshold_sensitivity_bar.png", dpi=220, bbox_inches="tight")
plt.show()